Tasks - Linear Algebra 1 (Complex)

These tasks require combining multiple topics and implementing algorithms from scratch.
Topics: PCA, SVD, low-rank approximation, Normal Equation, Cholesky sampling,
PyTorch autograd, SVD-PCA equivalence, and LoRA-style weight compression.

Read each task description carefully before writing code.
Expected time per task: 10-20 minutes.

Task 1 - PCA from Scratch

Implement Principal Component Analysis using eigendecomposition.
PCA finds the directions of maximum variance in the data.

Steps:
1. Centre the data by subtracting the column mean
2. Compute the covariance matrix
3. Eigendecompose the covariance matrix
4. Sort eigenvectors by descending eigenvalue
5. Project the data onto the first principal component

In [ ]:
import numpy as np

np.random.seed(0)
mean = [3.0, 5.0]
cov  = [[2.0, 1.5], [1.5, 1.5]]
X = np.random.multivariate_normal(mean, cov, size=100)  # shape (100, 2)

# step 1: centre the data
X_centered = None  # X - column means
print(f'original mean: {X.mean(axis=0).round(4)}')
print(f'centred mean : {X_centered.mean(axis=0).round(6)}  (should be ~0)')

# step 2: covariance matrix -- formula: (1/(n-1)) * X_centered.T @ X_centered
n = X_centered.shape[0]
cov_matrix = None  # your code here
print(f'\ncovariance matrix:\n{cov_matrix.round(4)}')

# step 3: eigendecompose the covariance matrix
eigenvalues, eigenvectors = None  # np.linalg.eig(cov_matrix)

# step 4: sort by descending eigenvalue
sort_idx     = None  # argsort descending
eigenvalues  = eigenvalues[sort_idx]
eigenvectors = eigenvectors[:, sort_idx]
print(f'\neigenvalues (sorted) : {eigenvalues.round(4)}')
print(f'explained var ratio  : {(eigenvalues / eigenvalues.sum()).round(4)}')

# step 5: project data onto the first principal component (PC1)
PC1 = eigenvectors[:, 0]  # direction of maximum variance
X_projected = None  # X_centered @ PC1, shape (100,)
print(f'\nprojection shape       : {X_projected.shape}')
print(f'variance of projection : {X_projected.var():.4f}')
print(f'first eigenvalue       : {eigenvalues[0]:.4f}  (should match variance)')

Task 2 - SVD Decomposition and Reconstruction

Decompose a matrix using Singular Value Decomposition: A = U @ diag(S) @ V.T
U (left singular vectors) and V (right singular vectors) are orthogonal.
S (singular values) are non-negative, sorted in descending order.

Decompose A, reconstruct it, then compute the rank-1 approximation.

In [ ]:
import numpy as np

A = np.array([[3.0,  2.0,  2.0],
              [2.0,  3.0, -2.0]])

# compute full SVD
U, S, Vt = None  # np.linalg.svd(A, full_matrices=True)

print(f'A shape  : {A.shape}')
print(f'U shape  : {U.shape}   (left singular vectors)')
print(f'S shape  : {S.shape}   (singular values)')
print(f'Vt shape : {Vt.shape}  (V transposed)')
print(f'\nsingular values: {S}')

# reconstruct A from U, S, Vt
# build the full sigma matrix (same shape as A), fill diagonal with S
Sigma = np.zeros_like(A)
np.fill_diagonal(Sigma, S)
A_reconstructed = None  # U @ Sigma @ Vt

print(f'\nA original:\n{A}')
print(f'A reconstructed:\n{A_reconstructed.round(6)}')
print(f'reconstruction error: {np.linalg.norm(A - A_reconstructed):.2e}')

# rank-1 approximation: use only the largest singular value/vector pair
A_rank1 = None  # S[0] * np.outer(U[:, 0], Vt[0, :])
print(f'\nrank-1 approximation:\n{A_rank1.round(4)}')
print(f'rank-1 error: {np.linalg.norm(A - A_rank1):.4f}')

Task 3 - Low-Rank Approximation via SVD

A rank-k approximation keeps only the top k singular values and vectors.
This is the mathematical foundation of image compression and LoRA fine-tuning.
The Eckart-Young theorem guarantees this is the best rank-k approximation in Frobenius norm.

Implement the approximation function, test it at multiple values of k,
and compute how much of the total energy (variance) each k captures.

In [ ]:
import numpy as np

np.random.seed(42)
A = np.random.randn(30, 20)  # simulated weight matrix or image patch
U, S, Vt = np.linalg.svd(A, full_matrices=False)

def low_rank_approx(U, S, Vt, k):
    # return the rank-k approximation of A
    # use the first k columns of U, first k singular values, first k rows of Vt
    pass  # your code here

total_params = A.shape[0] * A.shape[1]

for k in [1, 5, 10, 20]:
    A_k       = None  # call low_rank_approx(U, S, Vt, k)
    error     = np.linalg.norm(A - A_k, 'fro')
    lora_params = k * (U.shape[0] + Vt.shape[1])
    ratio     = lora_params / total_params
    print(f'k={k:2d}  Frobenius error={error:.4f}  param ratio={ratio:.2f}')

# cumulative explained energy (sum of S^2)
explained = (S**2).cumsum() / (S**2).sum()
print(f'\ncumulative energy captured:')
for k in [1, 5, 10, 20]:
    print(f'  k={k}: {explained[k-1]*100:.1f}%')

Task 4 - Normal Equation for Linear Regression

The Normal Equation solves linear regression in closed form:
w = (X^T X)^-1 X^T y

It finds the weight vector that minimises the mean squared error.
Steps: augment X with a bias column, compute X^T X, invert it, multiply by X^T y.

In [ ]:
import numpy as np

np.random.seed(7)
n = 50
X_raw = np.random.randn(n, 2)            # 50 samples, 2 features
true_w = np.array([2.0, 3.0])
true_b = 5.0
y = X_raw @ true_w + true_b + 0.1 * np.random.randn(n)

# add a bias column (column of 1s) to X
X = np.hstack([np.ones((n, 1)), X_raw])  # shape (50, 3)

# step 1: X^T X
XTX = None  # your code here

# step 2: inverse of X^T X
XTX_inv = None  # your code here

# step 3: X^T y
XTy = None  # your code here

# step 4: w = (X^T X)^-1 X^T y
w = None  # your code here

print(f'true  : bias={true_b}, w={true_w}')
print(f'normal eq: bias={w[0]:.4f}, w={w[1:].round(4)}')

# mean squared error
y_pred = X @ w
mse = None  # mean of (y - y_pred)^2
print(f'MSE: {mse:.6f}')

# verify with np.linalg.lstsq (numerically more stable than manual inverse)
w_lstsq, _, _, _ = None  # np.linalg.lstsq(X, y, rcond=None)
print(f'\nlstsq: {w_lstsq.round(4)}')
print(f'solutions match: {np.allclose(w, w_lstsq, atol=1e-4)}')

Task 5 - Cholesky Decomposition

For a positive definite matrix A, Cholesky gives A = L @ L.T where L is lower triangular.
This is used to efficiently sample from multivariate Gaussian distributions:
if z ~ N(0, I), then L @ z ~ N(0, Sigma) where Sigma = L @ L.T

Decompose a covariance matrix and use it to generate correlated samples.

In [ ]:
import numpy as np

Sigma = np.array([[4.0, 2.0, 0.5],
                  [2.0, 3.0, 1.0],
                  [0.5, 1.0, 2.0]])  # positive definite covariance

# compute Cholesky decomposition: Sigma = L @ L.T
L = None  # np.linalg.cholesky(Sigma)

# verify reconstruction
Sigma_recovered = None  # L @ L.T

print(f'Sigma:\n{Sigma}')
print(f'\nL (lower triangular):\n{L.round(4)}')
print(f'\nL @ L.T:\n{Sigma_recovered.round(4)}')
print(f'recovery check: {np.allclose(Sigma, Sigma_recovered)}')

# sample 1000 points from N(0, Sigma)
# generate independent standard normals, then multiply by L
np.random.seed(42)
z = np.random.randn(3, 1000)   # shape (3, 1000)
samples = None  # L @ z  -- shape (3, 1000)

# compute empirical covariance (should be close to Sigma)
empirical_cov = None  # np.cov(samples)

print(f'\nempirical covariance (1000 samples):\n{empirical_cov.round(2)}')
print(f'\ntrue Sigma:\n{Sigma}')

Task 6 - Autograd with PyTorch

PyTorch autograd automatically computes gradients of any differentiable function.
This is the engine behind backpropagation in neural network training.

Compute the gradient of a scalar function, then the gradient of the MSE loss.

In [ ]:
import torch

# --- part A: scalar gradient ---
# f(x) = x^3 + 2x^2 - 5x + 1
# analytical derivative: df/dx = 3x^2 + 4x - 5

x = torch.tensor(2.0, requires_grad=True)

# compute f(x)
f = None  # x**3 + 2*x**2 - 5*x + 1

# run backprop
None  # f.backward()

print(f'f(2.0)          = {f.item():.4f}')
print(f'df/dx at x=2    = {x.grad.item():.4f}')
print(f'analytical      = {3*2**2 + 4*2 - 5:.4f}  (3x^2 + 4x - 5 at x=2)')

# --- part B: MSE gradient ---
# L(w) = ||Xw - y||^2  (sum of squared residuals)
# analytical gradient: dL/dw = 2 * X.T @ (Xw - y)

X = torch.tensor([[1.0, 2.0],
                  [3.0, 4.0],
                  [5.0, 6.0]])
y = torch.tensor([7.0, 8.0, 9.0])
w = torch.tensor([0.5, 0.5], requires_grad=True)

residual = None  # X @ w - y
loss     = None  # (residual**2).sum()
None     # loss.backward()

print(f'\nloss       : {loss.item():.4f}')
print(f'dL/dw      : {w.grad}')
print(f'analytical : {(2 * X.T @ (X @ w.detach() - y)).numpy().round(4)}')

Task 7 - SVD and PCA Equivalence

PCA via eigendecomposition and PCA via SVD produce the same principal components.
The right singular vectors of X_centered equal the eigenvectors of the covariance matrix.
Singular values S relate to eigenvalues as: lambda_i = S_i^2 / (n - 1)

Verify this equivalence numerically on synthetic data.

In [ ]:
import numpy as np

np.random.seed(1)
X = np.random.randn(50, 4)  # 50 samples, 4 features
n = X.shape[0]

# centre the data
X_centered = None  # your code here

# method 1: PCA via eigendecomposition of covariance matrix
cov = None  # (1/(n-1)) * X_centered.T @ X_centered, shape (4, 4)
eigenvalues_eig, eigenvectors = None  # np.linalg.eig(cov)
# sort descending
idx = eigenvalues_eig.real.argsort()[::-1]
PC_eigen = eigenvectors[:, idx]  # principal components via eigen

# method 2: PCA via SVD of X_centered
U, S, Vt = None  # np.linalg.svd(X_centered, full_matrices=False)
PC_svd = Vt.T   # right singular vectors = principal components

# project data onto first 2 components using both methods
proj_eigen = None  # X_centered @ PC_eigen[:, :2]
proj_svd   = None  # X_centered @ PC_svd[:, :2]

# columns may have opposite signs but same magnitude
match = np.allclose(np.abs(proj_eigen), np.abs(proj_svd))
print(f'projections match (ignoring sign): {match}')

# verify eigenvalue relationship: lambda = S^2 / (n-1)
eigenvalues_sorted = np.sort(eigenvalues_eig.real)[::-1]
lambda_from_svd    = S**2 / (n - 1)
print(f'\neigenvalues (covariance) : {eigenvalues_sorted.round(4)}')
print(f'eigenvalues (from SVD)   : {lambda_from_svd.round(4)}')
print(f'match: {np.allclose(eigenvalues_sorted, lambda_from_svd)}')

Task 8 - LoRA-style Weight Compression

Low-Rank Adaptation (LoRA) replaces a weight matrix W (m x n) with two small matrices:
A (m x r) and B (r x n) so that W is approximated by A @ B, where r << min(m, n).
This reduces trainable parameters dramatically during fine-tuning of large models.

Decompose a weight matrix into LoRA factors using SVD, then measure the compression.

In [ ]:
import numpy as np

np.random.seed(5)
m, n = 64, 48       # weight matrix dimensions
r    = 4            # LoRA rank
W    = np.random.randn(m, n) * 0.1  # original weight matrix

# decompose W using SVD
U, S, Vt = None  # np.linalg.svd(W, full_matrices=False)

# keep only top-r components
U_r  = None  # first r columns of U, shape (m, r)
S_r  = None  # first r singular values
Vt_r = None  # first r rows of Vt, shape (r, n)

# define LoRA matrices A and B
# split the singular values symmetrically: A = U_r * sqrt(S_r), B = diag(sqrt(S_r)) @ Vt_r
A = None  # U_r * np.sqrt(S_r)           -- shape (m, r)
B = None  # np.diag(np.sqrt(S_r)) @ Vt_r -- shape (r, n)

# reconstruct using A @ B
W_approx = None  # your code here

# measure approximation error
error = np.linalg.norm(W - W_approx, 'fro')

print(f'W shape        : {W.shape}   total params: {W.size}')
print(f'A shape        : {A.shape}  params: {A.size}')
print(f'B shape        : {B.shape}  params: {B.size}')
print(f'total LoRA     : {A.size + B.size} params  ({(A.size + B.size)/W.size*100:.1f}% of original)')
print(f'\nFrobenius error: {error:.6f}')

# energy captured
total_energy   = (S**2).sum()
captured       = (S_r**2).sum()
print(f'energy captured: {captured/total_energy*100:.1f}%')